# 3. Joint construction: spectral gap, nonvanishing and the paper's conclusion

**Paper:** limit identification (Lemma 6.7 (`lem:z34-limits`)), selected error (Lemma 6.8 (`lem:z34-selected-error`)), common divisor (Lemma 6.14 (`prop:z34-divisor`)), nonvanishing (Lemma 6.15 (`lem:z34-nonvanishing`)), CMF Theorem 1 (Theorem 6.1 (`thm:cmf1`)) and linear independence (Theorem 6.4 (`thm:z34`)).

This notebook certifies the numerical inequalities with exact rational arithmetic and Arb interval arithmetic. It recomputes the density from the floor functions, so no saved output from another notebook is needed. The text separates analytic arguments from symbolic certificates and finite diagnostics.

The joint theorem uses the paper's conservative values $g>112.8075$, $\sigma<65.4516$, and divisor density $\ge30.3467$. These lead to $\delta>0.015$ and $\mu(A\zeta(3)+B\zeta(4))<66.3$. The separate improvement for $\zeta(3)$ uses a different arithmetic normalization and is outside this joint notebook.


Run every cell from top to bottom in a fresh Python kernel. This notebook uses only the local Python modules and matrix data included beside it; no other notebook needs to run first. Install the bundle's `requirements.txt` once. Saved output is a record of a completed run, while every certificate is recomputed when you run the cells.


In [1]:
from pathlib import Path
import sys
# Jupyter normally starts in the notebook folder; the bundle root also works.
_candidates = [Path.cwd(), Path.cwd() / "joint"]
NOTEBOOK_DIR = next((p.resolve() for p in _candidates
                     if (p / "certificates.py").is_file() and (p / "data" / "dual_companion.json").is_file()), None)
if NOTEBOOK_DIR is None:
    raise RuntimeError("Start this notebook from its joint/ folder or the bundle root.")
sys.path.insert(0, str(NOTEBOOK_DIR))
if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
print("Local joint-proof inputs found.")


Local joint-proof inputs found.


In [2]:
import json
import sympy as sp
from flint import arb, ctx
from ramanujantools import Matrix
from common import n, ROOT, matrix, determinant, regular_nonnegative
ctx.dps = 70
Q = Matrix(matrix("dual_matrices_zeta3_4.json","G")[:3,:3])
C = matrix("matrices_for_zeta3_4.json","C")
U = matrix("matrices_for_zeta3_4.json","U")

def ball(q):
    q = sp.Rational(q)
    return arb(int(q.p))/arb(int(q.q))

def enclosure(lo,hi):
    return arb(ball((lo+hi)/2),ball((hi-lo)/2))

def bracket(poly,lo,hi,steps=180):
    assert poly.eval(lo)<0<poly.eval(hi)
    for _ in range(steps):
        mid=(lo+hi)/2
        if poly.eval(mid)<0: lo=mid
        else: hi=mid
    return lo,hi

## A. Spectral rates — Lemma 6.8 (`lem:z34-selected-error`)
With $D(n)=\operatorname{diag}(1,n+1,(n+1)^2)$, the row state $\widehat Y_n=Y_nD(n)^{-1}$ has matrix $D(n)Q(n)D(n+1)^{-1}$. Every entry tends to a rational number; we obtain it from polynomial degrees and leading coefficients.

In [3]:
A = sp.zeros(3)
for i in range(3):
    for j in range(3):
        if Q[i,j] == 0: continue
        numerator,denominator = [sp.Poly(p,n) for p in sp.fraction(Q[i,j])]
        degree = numerator.degree()-denominator.degree()+i-j
        assert degree <= 0
        if degree == 0: A[i,j] = numerator.LC()/denominator.LC()
t = sp.Symbol("t")
chi = A.charpoly(t).as_poly()
assert chi.discriminant()<0
assert A.det()>0
lo,hi = bracket(chi,sp.Rational(7,10**24),sp.Rational(8,10**24))
r = enclosure(lo,hi)
a = -r.log()
b = (ball(A.det()).log()-r.log())/2
g = a+b
assert g > ball(sp.Rational("112.807539247454"))
print("Unique real root r:",r)
print("Recessive exponent -a:",-a)
print("Conjugate-pair modulus exponent b:",b)
print("Certified spectral gap g:",g)

Unique real root r: [7.202721273276511926282264010546996958113567659783099359e-24 +/- 7.83e-79]
Recessive exponent -a: [-53.2875833226202052673540400737687926618213110816266334281 +/- 6.96e-56]
Conjugate-pair modulus exponent b: [59.5199559248344750721435522284316210932854242854487175740 +/- 6.18e-56]
Certified spectral gap g: [112.807539247454680339497592302200413755106735367075351002 +/- 1.05e-55]


## B. Recessive-state identification — Lemma 6.7 (`lem:z34-limits`) and Appendix `app:zeta34-proof`
Notebook 2 proves that the residue state is
\[
\Psi_{n,j}=-\sum_{m>2n}\frac{d}{dm}\bigl((-m)^jR_n(m)\bigr),\qquad j=0,1,2,
\]
and satisfies the dual recurrence. Its initial state is $(1-2\zeta(3)+3\zeta(4),1-2\zeta(3),1)$, so it is not the zero solution.

Here is a coarse bound sufficient to identify its spectral mode, without a saddle-point evaluation of this sum. For $n\ge1$ and $2n\le t\le17n$, put
\[
B_n=(15n)^{14n}(16n)^{16n},\quad
D_n^*=(2n)^{11n+1}(3n)^{11n+1}(4n)^{8n+2}.
\]
The numerator roots lie in $[2n+1,16n]$ and $[n+1,17n]$. Thus $|P(t)|\le B_n$, $|P'(t)|\le(14/15+1)B_n<2B_n$. The denominator is at least $D_n^*$ and its logarithmic derivative is at most $(30n+4)/(2n)\le17$. Consequently $|R_n|\le B_n/D_n^*$ and $|R_n'|\le19B_n/D_n^*$, including at numerator zeros by the product rule.

For $t\ge17n$, $|R_n(t)|\le t^{-4}$ and $|R_n'(t)|\le(60n+4)t^{-5}$. The three differentiated tails therefore converge absolutely and grow at most polynomially in $n$. Combining the finite range and the tail gives
\[
\|\Psi_n\|\le\operatorname{poly}(n)e^{c_0n},\qquad
c_0=14\log15+16\log16-11\log2-11\log3-8\log4.
\]
The next cell verifies $c_0<b$. The standard exponential dichotomy for a difference system converging to the invertible, diagonalizable matrix $A$ says that a nonzero solution outside its one-dimensional recessive subspace has norm at least $e^{(b-o(1))n}$. Our bound excludes that possibility. Thus this explicit state is recessive, with exponential rate $-a$.

Polynomial coordinate scaling does not change these exponential rates.

### The asymptotic argument used here, with its hypotheses

Let $A_n=D(n)Q(n)D(n+1)^{-1}$. We have proved $A_n\to A$, every $A_n$ is invertible, and $A$ has eigenvalues $r,\lambda,\overline\lambda$ with $0<r<R=|\lambda|$. The claim needed is about **exponential rates**, not an unproved exact asymptotic expansion.

Here is a direct argument in this three-dimensional situation. A fixed real change of coordinates makes the limit matrix $\operatorname{diag}(r,R\mathcal R)$, where $\mathcal R$ is a planar rotation. Write the normalized dual solution in these coordinates as $(u_n,v_n)$, with $v_n\in\mathbb R^2$.

For each fixed $\delta>0$ and sufficiently small perturbations of that block matrix, the cone $\|v\|\ge\delta|u|$ is forward invariant. Its $v$ component expands by at least $R-\varepsilon$, where $\varepsilon$ can be arbitrarily small by starting sufficiently late. This follows from the triangle inequality in the two block equations; the strict inequality $R>r$ makes the cone invariant. Choose $\varepsilon$ so that $\log(R-\varepsilon)>c_0$. Our explicit upper bound excludes entry into that cone at any sufficiently late time. Since this works for every $\delta$, we obtain $v_n/u_n\to0$, and the first block equation gives $u_{n+1}/u_n\to r$. Thus the solution has rate $\log r$ and approaches the left $r$-eigenline. It is nonzero at every index because its initial value is nonzero and all transition matrices are invertible.

For the adjoint row system $x_{n+1}=x_nA_n^{-T}$, the pairing $x_n\widehat\Psi_n^T$ is constant. Use the dual change of coordinates, so this pairing is preserved. The planes annihilating $\widehat\Psi_n$ approach the fast coordinate plane. On those planes, the adjoint transition has norm ratio tending to $1/R$: the first coordinate is $o(1)$ times the planar norm and the limiting planar block is a rotation divided by $R$. Upper and lower bounds $1/R\pm\varepsilon$ therefore give rate $-\log R$ for every nonzero annihilating solution.

For a nonzero constant pairing, Cauchy's inequality gives adjoint growth at least $r^{-n+o(n)}$. The full transition norm in these fixed coordinates is at most $1/r+o(1)$, giving the matching upper rate. The same invariant-cone argument, now with a unique dominant first block $1/r>1/R$, gives convergence to that dominant line. In the original coordinates its row direction is the transpose of a **right** $r$-eigenvector of $A$. This is exactly the direction tested against the endpoint in the next section.

These arguments require convergence and separated moduli; they do not assume that $\sum_n\|A_n-A\|$ converges. For comparison with the standard literature, see the norm-growth and unique-dominant-direction results, Theorems 3–4, in [Borcea–Friedland–Shapiro, *Parametric Poincaré–Perron theorem with applications*](https://staff.math.su.se/shapiro/Articles/Poincare-Perron.pdf).

In [4]:
c0 = 14*arb(15).log()+16*arb(16).log()-11*arb(2).log()-11*arb(3).log()-8*arb(4).log()
assert c0 < b
assert sp.Rational(14,15)+1 < 2
assert sp.Rational(30+4,2) == 17
print("Elementary upper growth bound c0:",c0)
print("PASS: c0 is strictly below the dominant exponent b.")

Elementary upper growth bound c0: [51.47441332079971076703395486988673708609439560455694086444631383280655 +/- 6.21e-69]
PASS: c0 is strictly below the dominant exponent b.


In [5]:
from common import sm
R_readouts = Matrix([[12,0,0,0],[0,0,-6,3],[0,0,0,-2]])
K_pairing = (sm(matrix("dual_matrices_zeta3_4.json","E"))
             *sm(matrix("dual_matrices_zeta3_4.json","H"))).factor()
Z3,Z4 = sp.symbols("Z3 Z4")
G0 = Matrix([1-2*Z3+3*Z4,1-2*Z3,1])
assert R_readouts*K_pairing.subs({n:0})*G0 == Matrix([12,12*Z3,12*Z4])
print("PASS: exact initial-state readouts, including nonzero denominator 12.")


PASS: exact initial-state readouts, including nonzero denominator 12.


## C. Endpoint projection — Lemma 6.8 (`lem:z34-selected-error`)
Write $h_n=(1,10n+1,(10n+1)^2)^T/\Delta(n)$, where $\Delta(n)=\prod_{j=8}^{12}(jn+1)$. The common scalar $1/\Delta(n)$ cancels in every ratio. After the same normalization, $D(n)^{-1}h_n$ has leading direction $(1,10,100)^T$. Its coefficient along the dominant mode of the inverse-transpose system is nonzero if its pairing with a right eigenvector of $A$ at $r$ is nonzero.

We construct that eigenvector polynomially from an adjugate column. A nonzero resultant proves the pairing cannot vanish at any root of $\chi$, including $r$. Together with $\rho_qK(0)\Psi_0^T=12$, verified in Notebook 1, this proves the limit identification and denominator dominance required by the manuscript's error-rate argument. The ratio error is therefore at most $e^{-(g-o(1))n}$ for each fixed rational combination.

To make the mode argument explicit, let $\widehat W_n$ denote the product of the normalized matrices. The system is $A+O(1/n)$ with $A$ invertible and a single simple eigenvalue of smaller modulus. Its invariant splitting gives a one-dimensional recessive space and a two-dimensional growing space. In the inverse-transpose system, rows annihilating $\Psi_0^T$ have size at most $e^{-bn+o(n)}$, whereas any row with nonzero pairing with $\Psi_0^T$ has a dominant component of size $e^{an+o(n)}$. The normalized terminal vector tends, up to a scalar, to $(1,10,100)^T$. The nonzero projection proved below prevents that dominant denominator component from disappearing.

For the error row $\alpha\rho_q-A_0\rho_{p,3}-B_0\rho_{p,4}$ with $\alpha=A_0\zeta(3)+B_0\zeta(4)$, the initial pairing is zero by Notebook 1. For the denominator row it is $12$. Dividing therefore gives $e^{-(a+b)n+o(n)}$. This uses the invariant splitting, not merely the list of limiting eigenvalues.

In [6]:
adj = (t*sp.eye(3)-A).adjugate()
endpoint = sp.Matrix([1,10,100])
projection = sp.Poly((endpoint.T*adj[:,0])[0],t)
assert sp.resultant(chi.as_expr(),projection.as_expr(),t) != 0
print("PASS: endpoint has nonzero projection on the required mode.")

PASS: endpoint has nonzero projection on the required mode.


## D. Positive-sum growth — Lemma 6.9 (`prop:z34-q-iv`) and Lemma 6.14 (`prop:z34-divisor`)
The sum $\mathbf q_n=\sum_{k=0}^{14n}B_{n,k}$ is positive. Stirling's formula, uniformly over the $O(n)$ summands, gives the exponential growth $\sigma=\max_{0\le x\le14}s(x)$.

Here $s$ is the sum of five binomial entropy functions. Its derivative is the logarithm of
\[
\frac{(x+12)(x+13)(x+14)(14-x)(15-x)}{(x+2)(x+3)(x+4)(x+1)x}.
\]
This ratio strictly decreases from infinity to zero: each of the three first logarithmic derivative differences is negative, and all four remaining terms are negative. There is exactly one maximizer. We bracket its algebraic equation exactly and evaluate $s$ with outward-rounded intervals.

In [7]:
x=sp.Symbol("x")
f = sp.Poly(sp.prod(x+j for j in (2,3,4,1,0))-sp.prod(x+j for j in (12,13,14))*(14-x)*(15-x),x)
xl,xh = bracket(f,sp.Rational(0),sp.Rational(14))
xstar=enclosure(xl,xh)
L=lambda z:z*z.log()
sigma = sum(L(xstar+j)-L(arb(10))-L(xstar+j-10) for j in (12,13,14))
sigma += 2*L(arb(15))-L(xstar+1)-L(14-xstar)-L(xstar)-L(15-xstar)
assert sigma < ball(sp.Rational("65.451536565866"))
print("Maximizer:",xstar)
print("Growth exponent sigma:",sigma)
print("Required divisor density 76+sigma-g:",76+sigma-g)

Maximizer: [10.01471876885865451055118745082241453572312641227710782 +/- 8.64e-54]
Growth exponent sigma: [65.451536565865136726694433537699266758981023785501459 +/- 5.16e-52]
Required divisor density 76+sigma-g: [28.643997318410456387196841235498853003874288418426108 +/- 5.14e-52]


## E. Nonvanishing — Lemma 6.15 (`lem:z34-nonvanishing`)
Let $R$ have rows $\rho_q,\rho_{p,3},\rho_{p,4}$. The matrix of three consecutive readouts is
\[
R W_M(n)U(n)=R U(0)W_C(n).
\]
Its determinant is $\det(RU(0))\prod_{j<n}\det C(j)$. The first factor is nonzero below; the remaining factors are nonzero at every nonnegative integer by the exact regularity test.

Hence no nonzero linear combination of the three readout sequences vanishes at three consecutive indices. This proves the claim of Lemma 6.15 (`lem:z34-nonvanishing`). For fixed $(A,B)\ne(0,0)$, the three numerator/denominator pairs also have rank two, so at least one is distinct from any prescribed rational number. This is the nondegeneracy needed by the irrationality-measure criterion.

In [8]:
R=Matrix([[12,0,0,0],[0,0,-6,3],[0,0,0,-2]])
assert (R*U.subs({n:0})).det() != 0
num,den=sp.fraction(determinant(C))
assert regular_nonnegative(num) and regular_nonnegative(den)
print("PASS: every consecutive block of three readout vectors is independent.")

PASS: every consecutive block of three readout vectors is independent.


## F. Reproduce the paper's final calculations — Theorem 6.4 (`thm:z34`) and Theorem 6.1 (`thm:cmf1`)

The prime number theorem gives the normalization $D_n=6d_{19n}^4$ exponential rate $76$. We recompute the exact truncated divisor density $I_{40}$ and verify the rounded bound $I_{40}>30.3467$.

Using the paper's rounded constants, the integer representatives have a permissible height exponent
\[
h\le h_{\mathrm{up}}=76+65.4516-30.3467=111.1049<111.105.
\]
Then
\[
\delta\ge\frac{112.8075}{h_{\mathrm{up}}}-1>0.015,
\qquad
\mu\le\frac{112.8075}{112.8075-h_{\mathrm{up}}}<66.3.
\]
The exact fractions below certify these decimal comparisons. They do not replace the preceding analytic and arithmetic arguments.


In [9]:
from density import density_certificate, decimal_text
from fractions import Fraction
# Recompute rather than trusting another notebook's saved JSON.
density = density_certificate()
I40 = sp.Rational(str(density["integrals"][2]))
g_paper = sp.Rational("112.8075")
sigma_paper = sp.Rational("65.4516")
density_paper = sp.Rational("30.3467")
assert g > ball(g_paper)
assert sigma < ball(sigma_paper)
assert I40 > density_paper
h_upper = 76 + sigma_paper - density_paper
assert h_upper == sp.Rational("111.1049") < sp.Rational("111.105")
gap_lower = g_paper - h_upper
delta_lower = g_paper/h_upper - 1
mu_upper = g_paper/gap_lower
assert gap_lower > sp.Rational("1.70")
assert delta_lower > sp.Rational("0.015")
assert mu_upper < sp.Rational("66.3")
print("Paper divisor density >=", density_paper, "; computed truncation =", sp.N(I40,18))
print("Height exponent upper bound:", h_upper)
print("Positive gap lower bound:", gap_lower)
print("Approximation exponent >", sp.N(delta_lower,18), "> 0.015")
print("Irrationality-measure bound <", sp.N(mu_upper,18), "< 66.3")
report = {
    "paper_labels": ["thm:z34", "thm:cmf1"],
    "g_lower": str(g_paper), "sigma_upper": str(sigma_paper),
    "density_lower": str(density_paper), "truncated_density_exact": str(I40),
    "h_upper": str(h_upper), "gap_lower": str(gap_lower),
    "delta_lower_exact": str(delta_lower), "delta_lower_decimal": str(sp.N(delta_lower,20)),
    "mu_upper_exact": str(mu_upper), "mu_upper_decimal": str(sp.N(mu_upper,20)),
    "spectrum_and_endpoint_projection_certified": True,
    "nonvanishing_determinant_certified": True,
    "proof_inputs": ["exact readout identities and all-index integrality in Notebook 1",
                     "telescoping and valuation arguments in Notebook 2",
                     "recessive-state and endpoint argument in this notebook",
                     "Stirling's formula; prime number theorem; irrationality criterion"]
}
(ROOT/"results").mkdir(exist_ok=True)
(ROOT/"results"/"joint_paper_bounds.json").write_text(json.dumps(report,indent=2)+"\n")


Paper divisor density >= 303467/10000 ; computed truncation = 30.3467778194919024
Height exponent upper bound: 1111049/10000
Positive gap lower bound: 8513/5000
Approximation exponent > 0.0153242566259453903 > 0.015
Irrationality-measure bound < 66.2560202043932809 < 66.3


1422

## G. Integer contradiction — Theorem 6.4 (`thm:z34`)

Suppose an integer relation $A_0+A_3\zeta(3)+A_4\zeta(4)=0$ holds, with coefficients not all zero. Define the integer triple
\[
(\widetilde q_n,\widetilde p_{3,n},\widetilde p_{4,n})=\frac{6d_{19n}^4}{\mathcal D_n}(\mathbf q_n,\mathbf p_{3,n},\mathbf p_{4,n}),
\]
where $\mathcal D_n$ is the common divisor from the finite prime bounds in Notebook 2. Notebook 1 proves integrality, and Notebook 2 justify the two presentations and the divisor.

The integer
\[
L_n=A_0\widetilde q_n+A_3\widetilde p_{3,n}+A_4\widetilde p_{4,n}
=A_3(\widetilde p_{3,n}-\zeta(3)\widetilde q_n)+A_4(\widetilde p_{4,n}-\zeta(4)\widetilde q_n)
\]
has absolute value at most $e^{(h-g+o(1))n}$ times a fixed constant. Since the certified inequality is $h<g$, it tends to zero. Thus $L_n=0$ for every sufficiently large $n$.

But the determinant argument above says a nonzero fixed combination of the three readouts cannot vanish at three consecutive indices. Multiplying each readout triple by its nonzero scalar normalization does not change that statement. This is a contradiction. Therefore **$1,\zeta(3),\zeta(4)$ are linearly independent over $\mathbb Q$**.

This last argument uses only that integers tending to zero are eventually zero. The measure calculation above gives the manuscript’s stated bound for each fixed nonzero rational pair $(A,B)$.

**Conclusion (Theorem 6.4 (`thm:z34`) and Theorem 6.1 (`thm:cmf1`)).** The verified algebraic and arithmetic premises, together with the stated analytic arguments, prove that $1,\zeta(3),\zeta(4)$ are linearly independent over $\mathbb Q$. The joint construction gives $\delta>0.015$ and $\mu(A\zeta(3)+B\zeta(4))<66.3$ for every fixed rational $(A,B)\ne(0,0)$.

The symbolic residuals, exact floor integrations and interval inequalities are checked by Python. Stirling's formula, the prime number theorem, the asymptotic difference-system argument and the irrationality criterion remain mathematical theorem inputs. This Python supplement does not write or execute a Lean formalization.